# utils

> Tools for examining and debugging cogitarelink modules during development in Solveit or other jupyter like environments that have LLM co-pilots.

In [ ]:
#| default_exp utils

In [ ]:
#| hide
from nbdev.showdoc import *

## Usage with Solveit

These utilities are designed to help with development and debugging in solveit dialogs. They provide flexible options for loading source code into your dialog context.

### Loading Individual Modules

For targeted analysis when you're focused on specific modules:

```python
from cogitarelink.utils import load_module_source

# Load just the modules you need
core_py = load_module_source("core")
vocab_py = load_module_source("vocabulary")

# You can also load modules by full name
tools_py = load_module_source("cogitarelink.tools", full_name=True)
```

### Loading All Modules

When you need a comprehensive view of the codebase:

```python
from cogitarelink.utils import load_all_modules

# Load all core modules at once
module_sources = load_all_modules()

# Access individual module sources
print(f"Core module has {len(module_sources['core_py'])} characters")
```

### Exploring Module Structure

To get a quick overview of what's available in each module:

```python
from cogitarelink.utils import display_module_structure

# Show structure of a specific module
display_module_structure("navigation")

# Show structure of all modules
display_module_structure()
```

### Tips for Effective Use with Solveit

1. Load only what you need to preserve context window space
2. Use `display_module_structure()` first to understand what's available
3. When sharing code with solveit, include relevant module sources in context variables
4. For deep refactoring tasks, load all modules to help solveit understand interdependencies

This module provides utilities for loading and exploring the source code of cogitarelink modules. These tools are particularly useful for development, refactoring, and debugging within solveit dialogs and other interactive environments.

The utilities help developers:
- Load module source code into Python variables
- Examine module structure and contents
- Facilitate code analysis and refactoring
- Support debugging in interactive environments

These utilities are designed for development purposes and are not intended to be part of the public API.

In [ ]:
#| export
import inspect
import importlib
import sys
import os
from pathlib import Path
from typing import Dict, List, Optional, Union
from IPython.display import display, Markdown

## Loading Module Source Code

The `load_module_source` function retrieves the source code of a specified module:

```python
source = load_module_source("core")
print(f"Source code length: {len(source)}")
```

This is particularly useful when you need to analyze or debug code from a specific module.

In [ ]:
#| export
def load_module_source(module_name:str, # Name of the module to load (e.g., "core", "vocabulary")
                      full_name:bool=False # Whether the name includes the package prefix
                     ) -> str:
    """Load the source code of a module as a string
    
    Args:
        module_name: Name of the module to load
        full_name: Whether the name includes the package prefix
        
    Returns:
        The source code as a string
    """
    # Handle module name with or without package prefix
    if not full_name:
        full_module_name = f"cogitarelink.{module_name}"
    else:
        full_module_name = module_name
    
    try:
        # Import the module
        module = importlib.import_module(full_module_name)
        
        # Get the file path
        file_path = inspect.getfile(module)
        
        # Read the source code
        with open(file_path, 'r') as f:
            source = f.read()
            
        print(f"Loaded {module_name} from {file_path} ({len(source)} characters)")
        return source
    
    except Exception as e:
        print(f"Error loading {module_name}: {e}")
        return ""

## Loading All Modules

To load all core modules at once, use the `load_all_modules` function:

```python
sources = load_all_modules()
for name, source in sources.items():
    print(f"{name}: {len(source)} characters")
```

This creates a dictionary mapping module variable names (e.g., `core_py`) to their source code.

In [ ]:
#| export
def load_all_modules() -> Dict[str, str]:
    """Load the source code of all cogitarelink modules
    
    Returns:
        A dictionary mapping module names to their source code
    """
    core_modules = ["core", "vocabulary", "navigation", "dataset", "tools", "workflows"]
    sources = {}
    
    for module in core_modules:
        source = load_module_source(module)
        if source:
            sources[f"{module}_py"] = source
    
    return sources

## Displaying Module Structure

To get a quick overview of what's in a module, use `display_module_structure`:

```python
# Show structure of a specific module
display_module_structure("core")

# Show all modules
display_module_structure()
```

This displays the classes and functions defined in each module.

In [ ]:
#| export
def display_module_structure(module_name:str=None, # Module to display (or None for all)
                           show_functions:bool=True # Whether to list functions
                          ) -> None:
    """Display the structure of a module or all modules
    
    Args:
        module_name: Module to display, or None for all modules
        show_functions: Whether to list functions in the module
    """
    if module_name:
        modules = [module_name]
    else:
        modules = ["core", "vocabulary", "navigation", "dataset", "tools", "workflows"]
    
    for mod_name in modules:
        try:
            # Import the module
            full_name = f"cogitarelink.{mod_name}"
            module = importlib.import_module(full_name)
            
            # Get file info
            file_path = inspect.getfile(module)
            
            # Get all objects in the module
            objects = inspect.getmembers(module)
            
            # Filter to just functions and classes if requested
            if show_functions:
                functions = [name for name, obj in objects 
                           if inspect.isfunction(obj) and not name.startswith('_')]
                classes = [name for name, obj in objects 
                         if inspect.isclass(obj) and not name.startswith('_')]
            
            # Create markdown
            md = [f"## Module: {mod_name}"]
            md.append(f"File: {file_path}")
            
            if show_functions:
                if classes:
                    md.append("\n### Classes:")
                    for cls in sorted(classes):
                        md.append(f"- `{cls}`")
                
                if functions:
                    md.append("\n### Functions:")
                    for func in sorted(functions):
                        md.append(f"- `{func}`")
            
            display(Markdown("\n".join(md)))
            
        except Exception as e:
            print(f"Error displaying structure for {mod_name}: {e}")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()